<a href="https://colab.research.google.com/github/kaushik1506/ai-tech-gen-student-performance/blob/main/student_performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [86]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from google.colab import files

In [87]:
df = pd.read_csv('Student_Performance.csv')


In [88]:
df

,student_id,age,gender,school_type,parent_education,study_hours,attendance_percentage,internet_access,travel_time,extra_activities,study_method,math_score,science_score,english_score,overall_score,final_grade
0,1,14,male,public,post graduate,3.1,84.3,yes,<15 min,yes,notes,42.7,55.4,57.0,53.1,e
1,2,18,female,public,graduate,3.7,87.8,yes,>60 min,no,textbook,57.6,68.8,64.8,61.3,d
2,3,17,female,private,post graduate,7.9,65.5,no,<15 min,no,notes,84.8,95.0,79.2,89.6,b
3,4,16,other,public,high school,1.1,58.1,no,15-30 min,no,notes,44.4,27.5,54.7,41.6,e
4,5,16,female,public,high school,1.3,61.0,yes,30-60 min,yes,group study,8.9,32.7,30.0,25.4,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,12047,17,female,public,phd,1.8,55.2,yes,15-30 min,no,mixed,55.8,48.5,46.7,46.1,e
24996,1102,16,female,private,diploma,2.7,97.1,yes,<15 min,no,coaching,64.8,48.2,52.3,56.5,d
24997,4422,19,other,private,post graduate,1.0,63.0,yes,<15 min,no,group study,50.5,20.3,36.1,36.7,f
24998,7858,14,male,private,diploma,1.0,69.4,yes,15-30 min,yes,group study,13.0,34.2,7.3,34.1,f


In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   student_id             25000 non-null  int64  
 1   age                    25000 non-null  int64  
 2   gender                 25000 non-null  object 
 3   school_type            25000 non-null  object 
 4   parent_education       25000 non-null  object 
 5   study_hours            25000 non-null  float64
 6   attendance_percentage  25000 non-null  float64
 7   internet_access        25000 non-null  object 
 8   travel_time            25000 non-null  object 
 9   extra_activities       25000 non-null  object 
 10  study_method           25000 non-null  object 
 11  math_score             25000 non-null  float64
 12  science_score          25000 non-null  float64
 13  english_score          25000 non-null  float64
 14  overall_score          25000 non-null  float64
 15  fi

In [90]:
df.describe()

,student_id,age,study_hours,attendance_percentage,math_score,science_score,english_score,overall_score
count,25000.00000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000
mean,7493.04380,16.482760,4.253224,75.084084,63.785944,63.745320,63.681948,64.006172
std,4323.56215,1.703895,2.167541,14.373171,20.875262,20.970529,20.792693,18.932025
min,1.00000,14.000000,0.500000,50.000000,0.000000,0.000000,0.000000,14.500000
25%,3743.75000,15.000000,2.400000,62.800000,48.300000,48.200000,48.300000,49.000000
50%,7461.50000,16.000000,4.300000,75.100000,64.100000,64.100000,64.200000,64.200000
75%,11252.00000,18.000000,6.100000,87.500000,80.000000,80.000000,80.000000,79.000000
max,15000.00000,19.000000,8.000000,100.000000,100.000000,100.000000,100.000000,100.000000


In [91]:
df['final_grade'].value_counts()

,count
final_grade,
d,6311
c,6161
e,5672
f,2955
b,2696
a,1205


In [92]:
df.isnull().sum()

,0
student_id,0
age,0
gender,0
school_type,0
parent_education,0
study_hours,0
attendance_percentage,0
internet_access,0
travel_time,0
extra_activities,0


In [93]:
df = df.drop(columns=['student_id'], errors='ignore')
df = df.drop_duplicates()
df = df.dropna(thresh=len(df)*0.5, axis=1)

In [94]:
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [95]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='final_grade', hue='final_grade', palette='viridis', legend=False, order=['a', 'b', 'c', 'd', 'e', 'f'])
plt.savefig('grade_distribution.png', bbox_inches='tight', dpi=300)
plt.close()

files.download('grade_distribution.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [96]:
sample_df = df.select_dtypes(include=['float64', 'int64']).join(df['final_grade']).sample(n=min(500, len(df)), random_state=42)
sns.pairplot(sample_df, hue='final_grade', palette='viridis')
plt.savefig('pairplot_grades.png', bbox_inches='tight', dpi=300)
plt.close()

files.download('pairplot_grades.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [97]:
numeric_df = df.select_dtypes(include=['float64', 'int64'])
corr = numeric_df.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.savefig('correlation_heatmap.png', bbox_inches='tight', dpi=300)
plt.close()

files.download('correlation_heatmap.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [98]:
X = df.drop(columns=['final_grade', 'overall_score'], errors='ignore')
X = pd.get_dummies(X, drop_first=True)
y = df['final_grade']

In [99]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [100]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [101]:
y_pred = model.predict(X_test)

print(accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues')
plt.savefig('confusion_matrix.png', bbox_inches='tight', dpi=300)
plt.close()

files.download('confusion_matrix.png')

0.7576666666666667
              precision    recall  f1-score   support

           a       0.80      0.69      0.74       144
           b       0.65      0.59      0.62       328
           c       0.75      0.81      0.78       739
           d       0.76      0.77      0.77       754
           e       0.76      0.78      0.77       676
           f       0.85      0.77      0.81       359

    accuracy                           0.76      3000
   macro avg       0.76      0.73      0.75      3000
weighted avg       0.76      0.76      0.76      3000



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [102]:
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20]
}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, n_jobs=-1)
grid_search.fit(X_train, y_train)

model = grid_search.best_estimator_
y_pred_best = model.predict(X_test)

print(grid_search.best_params_)
print(accuracy_score(y_test, y_pred_best))

{'max_depth': None, 'n_estimators': 100}
0.7576666666666667


In [103]:
metrics_RF = f"""
Classification Pipeline Report (Grades)
------------------------------
Dataset Shape (Rows, Columns): {df.shape}
Features Selected: {len(X.columns)} features (after one-hot encoding)
Target Variable: {y.name}

Baseline Model Accuracy: {accuracy_score(y_test, y_pred):.4f}
Improved Model Accuracy: {accuracy_score(y_test, y_pred_best):.4f}
Best Hyperparameters: {grid_search.best_params_}
"""
print(metrics_RF)


Classification Pipeline Report (Grades)
------------------------------
Dataset Shape (Rows, Columns): (15000, 15)
Features Selected: 24 features (after one-hot encoding)
Target Variable: final_grade

Baseline Model Accuracy: 0.7577
Improved Model Accuracy: 0.7577
Best Hyperparameters: {'max_depth': None, 'n_estimators': 100}

